# «A los 18 llamé a la puerta» en Google Colab (scripts locales)

El mismo flujo que en el PC (`doblaje_scripts/PUERTA.md`), pero en Colab. Todo se guarda en **Drive**, en
`Mi unidad/mangavid/puerta`: si se corta la sesión, vuelves a ejecutar las celdas 1-3 y el paso donde estabas **sigue donde lo dejó**.

1. *Entorno de ejecución → Cambiar tipo → T4 GPU*.
2. Pega `notebooks/mantener_colab.js` en la consola (F12) para que no se apague la sesión.
3. Ejecuta las celdas en orden. Las largas van en segundo plano: si paras el vigilante, la tarea sigue.

In [ ]:
#@title 1. Drive, código e instalación
import os, getpass, subprocess, time, shutil, datetime, glob
from google.colab import drive
from IPython.display import clear_output, Image, display
drive.mount("/content/drive")
CARPETA = "/content/drive/MyDrive/mangavid"  #@param {type:"string"}
REPO = "/content/doblaje"
if not os.path.exists(REPO):
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    token = token or getpass.getpass("Token de GitHub: ")
    !git clone -q https://{token}@github.com/statsuya500/yousef-research-engine.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q -r requirements.txt

# Los scripts escriben en /mnt/c/videos/puerta (como en WSL): aquí esa ruta apunta a Drive.
os.makedirs(CARPETA + "/puerta", exist_ok=True)
os.makedirs("/mnt/c", exist_ok=True)
if not os.path.islink("/mnt/c/videos"):
    os.symlink(CARPETA, "/mnt/c/videos")
H = "/mnt/c/videos/puerta"
if not os.path.exists(H + "/historia.txt"):
    shutil.copy("historias/puerta/historia_local.txt", H + "/historia.txt")
print("✅ Listo. Resultados en", CARPETA + "/puerta")

In [ ]:
#@title 2. Utilidades (lanzar en segundo plano y vigilar)
def lanzar(orden, log, sin_conexion=True):
    env = dict(os.environ, PYTHONPATH=REPO)
    if sin_conexion:
        env["HF_HUB_OFFLINE"] = "1"
    else:
        env.pop("HF_HUB_OFFLINE", None)
    proceso = subprocess.Popen(orden, cwd=REPO, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT, text=True)
    print(f"▶ En marcha (proceso {proceso.pid}). Log: {log}")
    return proceso

def vigilar(proceso, log, cada=30, lineas=12):
    inicio = time.time()
    while proceso.poll() is None:
        clear_output(wait=True)
        print(f"⏳ {datetime.datetime.now():%H:%M:%S} · {(time.time() - inicio) / 60:.0f} min")
        print("─" * 60)
        try:
            print("".join(open(log, errors="replace").readlines()[-lineas:]))
        except FileNotFoundError:
            pass
        time.sleep(cada)
    clear_output(wait=True)
    print("".join(open(log, errors="replace").readlines()[-lineas:]))
    print("✅ Terminado" if proceso.returncode == 0 else f"❌ Error {proceso.returncode}: mira el log")

def paso(script, *args, log=None, sin_conexion=True):
    log = log or f"/content/{script}.log"
    vigilar(lanzar(["python", f"doblaje_scripts/{script}.py", *args], log, sin_conexion), log)

In [ ]:
#@title 3. Descargar los modelos (5-10 min; hay que repetirlo en cada sesión nueva)
p = lanzar(["python", "-m", "mangavid", "modelos"], "/content/modelos.log", sin_conexion=False)
vigilar(p, "/content/modelos.log")

## 4. Pasos (uno detrás de otro)

In [ ]:
#@title Paso 1: duración de la voz de cada escena (~10 min, necesita internet)
paso("puerta_duraciones", sin_conexion=False)

In [ ]:
#@title Paso 2: calibración (pruebas de cada personaje y lugar)
CASOS = ""  #@param {type:"string"}
#@markdown Vacío = todos. Para repetir solo algunos: `dario-sudadera isabel-gala`
paso("puerta_calibrar", *CASOS.split())
paso("puerta_filtro", "calibrar", log="/content/filtro_calibrar.log")

In [ ]:
#@title Ver la calibración
FILTRO = "dario-sudadera"  #@param {type:"string"}
for f in sorted(glob.glob(f"{H}/calibracion/{FILTRO}*.png")):
    print(os.path.basename(f)); display(Image(f, width=400))

In [ ]:
#@title Paso 3: generar las ~1.780 imágenes (varias horas; reanudable)
paso("puerta_generar")

In [ ]:
#@title Paso 4: filtrar y montar
paso("puerta_filtro")
paso("puerta_montar")

In [ ]:
#@title Paso 5: vídeo final (necesita internet para la voz)
paso("render_sistema", H, H + "/a_los_18.mp4", H + "/trabajo/tramos", log="/content/render.log", sin_conexion=False)